In [1]:
import pandas as pd
import numpy as np
import torch

import matplotlib.pyplot as plt
import plotly.graph_objects as go

import urllib3 # to download data directly from web

# CH4

# Vostok CH4

[Overview of Vostok data sets](https://www.ncei.noaa.gov/access/paleo-search/study/2453)  
Called "Atmospheric Methane Data"

In [2]:
# define link to data
url = "https://www.ncei.noaa.gov/pub/data/paleo/icecore/antarctica/vostok/ch4nat-noaa.txt"

# use urllib3 package to download .txt data directly from web
# Creating a PoolManager instance for sending requests.
http = urllib3.PoolManager()

# Sending a GET request and getting back response as HTTPResponse object.
resp = http.request("GET", url)

# create new (empty) text file ('w' stands for write mode)
file = open('../data/vostok/ch4/downloaded_ch4nat-noaa.txt', 'w')
# populate text file with our data
file.write(resp.data.decode('utf-8-sig'))
# close file (write mode)
file.close()

# open file in read mode
file = open("../data/vostok/ch4/downloaded_ch4nat-noaa.txt", "r")


for index, line in enumerate(file):
    if not "#" in line:
        if line.split() == ['gas_ageBP', 'CH4']:
            # Initialise pd dataframe
            data = pd.DataFrame(columns = ('gas_ageBP', 'CH4'))
        else:
            data.loc[index] = [float(i) for i in line.split()]

# reset the index
data = data.reset_index(drop = True)

In [3]:
data

,gas_ageBP,CH4
0,2347.0,668.0
1,3634.0,636.0
2,3833.0,595.0
3,6225.0,588.0
4,6614.0,574.0
...,...,...
452,410793.0,623.0
453,412182.0,626.0
454,414080.0,653.0
455,415452.0,678.0


In [4]:
x_reg_400kyr = np.arange(start = 0, stop = 400000 + 1, step = 1000)
y_reg_400kyr = np.interp(x = x_reg_400kyr, xp = data["gas_ageBP"], fp = data["CH4"])

CH4_timeseries_400kyr_vostok = torch.flip(torch.tensor(y_reg_400kyr), dims = [0])

# Non-uniqueness

In [5]:
def affirm_uniqueness(ts):
    # Check for duplicates and add noise 
    dupes = ts.shape[0] - torch.unique(ts.to(torch.float32)).shape[0]
    print("There are ", dupes, " duplicates in the timeseries.")
    if dupes > 0:
        noise_level = 0.001
        while dupes > 0:
            ts = ts + torch.randn(ts.shape[0]) * noise_level
            # recalculate dupes
            dupes = ts.shape[0] - torch.unique(ts.to(torch.float32)).shape[0]
            print("Now we have ", dupes, " dupes.")
    return ts

In [6]:
CH4_timeseries_400kyr_vostok = affirm_uniqueness(CH4_timeseries_400kyr_vostok)

There are  3  duplicates in the timeseries.
Now we have  1  dupes.
Now we have  0  dupes.


# Export

In [7]:
torch.save(CH4_timeseries_400kyr_vostok, '../data/vostok/ch4/CH4_vostok_400kyr_timeseries.pt')

# EPICA CH4

- [see integrated README](https://www.ncei.noaa.gov/pub/data/paleo/icecore/antarctica/epica_domec/edc-ch4-2008.txt)
- format of txt file is slightly different so required different code to process

In [8]:
# define link to data
url = "https://www.ncei.noaa.gov/pub/data/paleo/icecore/antarctica/epica_domec/edc-ch4-2008.txt"

# use urllib3 package to download .txt data directly from web
# Creating a PoolManager instance for sending requests.
http = urllib3.PoolManager()

# Sending a GET request and getting back response as HTTPResponse object.
resp = http.request("GET", url)

# create new (empty) text file ('w' stands for write mode)
file = open('../data/epica/ch4/downloaded_edc-ch4-2008.txt', 'w')
# populate text file with our data
file.write(resp.data.decode('utf-8-sig'))
# close file (write mode)
file.close()

# open file in read mode
file = open("../data/epica/ch4/downloaded_edc-ch4-2008.txt", "r")

data = pd.DataFrame(columns = ('depth', 'gas_ageBP', 'CH4', '1s', 'lab'))

for index, line in enumerate(file):
    # Needs to be manual is file does not have #
    # last row is empty so stop there
    if (index > 153) & (index < 2257): 
        data.loc[index - 153] = [i for i in line.split()]

# select the two columns we need
# change integers in pandas dataframe to floats
data = data[["gas_ageBP", "CH4"]].astype('float64')

# reset the index
data = data.reset_index(drop = True)

In [9]:
data

,gas_ageBP,CH4
0,13.0,907.0
1,126.0,784.0
2,130.0,762.0
3,151.0,710.0
4,184.0,727.0
...,...,...
2098,794938.0,428.0
2099,796320.0,418.0
2100,797277.0,396.0
2101,798417.0,458.0


In [10]:
# target x at which we wan't to interpolate
x_reg = np.arange(start = 0, stop = 800000 + 1, step = 1000)

# np interpolate
y_reg = np.interp(x = x_reg, xp = data["gas_ageBP"], fp = data["CH4"])

# flip order 
CH4_EPICA_timeseries = torch.flip(torch.tensor(y_reg), dims = [0])
CH4_EPICA_timeseries

tensor([399.0000, 422.8652, 435.3211, 402.3678, 420.3155, 427.5514, 435.4229,
        465.7256, 466.0240, 492.0247, 540.1469, 554.1365, 608.1333, 708.4449,
        680.4717, 661.6810, 647.4412, 633.5771, 623.7696, 630.8615, 613.8941,
        606.6577, 679.8167, 630.6667, 617.8182, 579.0472, 563.0112, 565.9731,
        539.3027, 644.9854, 620.6402, 569.0148, 515.0449, 514.5248, 463.5833,
        530.7500, 552.0800, 525.6242, 473.9161, 461.3731, 461.4847, 427.2115,
        498.0000, 473.1584, 430.8215, 424.9292, 410.2893, 424.7521, 410.0406,
        420.1966, 398.6345, 400.0444, 397.8588, 404.9083, 415.1595, 424.5616,
        430.7525, 416.1193, 406.6991, 407.8162, 406.7811, 407.3987, 458.8766,
        556.9697, 538.5403, 528.8343, 522.9084, 524.0888, 531.8153, 538.2411,
        547.9816, 533.0272, 541.9973, 531.2774, 549.0719, 484.9968, 569.5650,
        565.7654, 518.8335, 458.9724, 405.1874, 397.7994, 402.3943, 408.5902,
        489.6716, 570.8693, 583.7051, 596.8605, 581.1219, 589.57

## Non-uniqueness

In [11]:
CH4_EPICA_timeseries = affirm_uniqueness(CH4_EPICA_timeseries)

There are  4  duplicates in the timeseries.
Now we have  0  dupes.


## Export

In [12]:
torch.save(CH4_EPICA_timeseries, '../data/epica/ch4/CH4_epica_800kyr_timeseries.pt')